<a href="https://colab.research.google.com/github/AarnavSawant/KVCompass/blob/main/benchmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# KVCompass Benchmarks Notebook

This notebook is the benchmark collection half of the original `KVCompass.ipynb`. Run these cells to set up Colab, authenticate with Hugging Face, generate the assignment sweep configs, and execute the benchmark runs that write artifacts to Google Drive.


## Shared Experiment Plan

**Benchmark**
- `RULER`
- `fraction: 0.02`
- context lengths: `4096`, `8192`

**Task categories**
- Needle In A Haystack: `niah`
- Question Answering: `qa`
- Multi-Hop Tracing: `vt`
- Aggregation: `cwe`, `fwe`

**Methods in the core matrix**
- `no_compression @ 1.0`
- `snapkv @ 0.5`
- `expected_attention @ 0.5`
- `knorm @ 0.5`
- `tova @ 0.5`
- `streaming_llm @ 0.5`

**Team split**
- Tony: `niah` compressed-method runs at `4096` and `8192`
- Will: `qa` compressed-method runs at `4096` and `8192`
- Grady: `vt` compressed-method runs at `4096` and `8192`
- Jamez: all `no_compression` baselines across all categories and both context lengths
- Aarnav: `aggregation` compressed-method runs at `4096` and `8192`, plus final aggregation

**Core matrix size**
- `40` total benchmark runs
- `32` compressed-method runs across the four category owners
- `8` baseline runs for Jamez


In [ ]:
# Shared setup: clone/refresh the repo, switch to the demo branch, mount Drive, and install dependencies.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BRANCH_NAME = 'main'
repo_dir = Path('/content/KVCompass')
if not repo_dir.exists():
    !git clone https://github.com/AarnavSawant/KVCompass.git /content/KVCompass

%cd /content/KVCompass
!git fetch origin
!git checkout "$BRANCH_NAME"
!git pull origin "$BRANCH_NAME"
!nvidia-smi
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -e .


In [ ]:
# Shared auth: set HF_TOKEN from Colab secrets and verify the login.
from google.colab import userdata
from huggingface_hub import whoami
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print(whoami())


In [ ]:
# Shared values used by all run cells.
from pathlib import Path

MODEL_NAME = 'Qwen/Qwen3-8B'
FRACTION = 0.25
TORCH_DTYPE = 'bfloat16'
SHARED_RESULTS_DIR = Path('/content/drive/MyDrive/KVCompass')
SHARED_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Shared results dir:', SHARED_RESULTS_DIR)


## Tony: Needle In A Haystack

Run the next two cells only. This assignment covers the benchmark-backed `niah` slice for the compressed methods at `4096` and `8192`.


In [ ]:
# Write the sweep config for Assignment 1.
from pathlib import Path

config_text = f"""
sweep:
  name: assignment_1
  model: {MODEL_NAME}
  device: auto
  torch_dtype: {TORCH_DTYPE}
  methods_config_path: configs/methods.yaml
  output_dir: {SHARED_RESULTS_DIR.as_posix()}/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: needle_in_a_haystack_4k
      dataset: ruler
      data_dir: "4096"
      task_prefixes: ["niah"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - tova
        - streaming_llm
      budgets:
        default: [0.5]
    - name: needle_in_a_haystack_8k
      dataset: ruler
      data_dir: "8192"
      task_prefixes: ["niah"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - tova
        - streaming_llm
      budgets:
        default: [0.5]
"""
Path('configs/benchmark_sweeps.assignment_1.yaml').write_text(config_text, encoding='utf-8')
print(Path('configs/benchmark_sweeps.assignment_1.yaml').read_text())


In [ ]:
# Run Assignment 1.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.assignment_1.yaml


## Will: Question Answering

Run the next two cells only. This assignment covers the benchmark-backed `qa` slice for the compressed methods at `4096` and `8192`.


In [ ]:
# Write the sweep config for Assignment 2.
from pathlib import Path

config_text = f"""
sweep:
  name: assignment_2
  model: {MODEL_NAME}
  device: auto
  torch_dtype: {TORCH_DTYPE}
  methods_config_path: configs/methods.yaml
  output_dir: {SHARED_RESULTS_DIR.as_posix()}/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: question_answering_4k
      dataset: ruler
      data_dir: "4096"
      task_prefixes: ["qa"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - tova
        - streaming_llm
      budgets:
        default: [0.5]
    - name: question_answering_8k
      dataset: ruler
      data_dir: "8192"
      task_prefixes: ["qa"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - tova
        - streaming_llm
      budgets:
        default: [0.5]
"""
Path('configs/benchmark_sweeps.assignment_2.yaml').write_text(config_text, encoding='utf-8')
print(Path('configs/benchmark_sweeps.assignment_2.yaml').read_text())


In [ ]:
# Run Assignment 2.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.assignment_2.yaml


## Grady: Multi-Hop Tracing

Run the next two cells only. This assignment covers the benchmark-backed `vt` slice for the compressed methods at `4096` and `8192`.


In [ ]:
# Write the sweep config for Assignment 3.
from pathlib import Path

config_text = f"""
sweep:
  name: assignment_3
  model: {MODEL_NAME}
  device: auto
  torch_dtype: {TORCH_DTYPE}
  methods_config_path: configs/methods.yaml
  output_dir: {SHARED_RESULTS_DIR.as_posix()}/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: multi_hop_tracing_4k
      dataset: ruler
      data_dir: "4096"
      task_prefixes: ["vt"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - tova
        - streaming_llm
      budgets:
        default: [0.5]
    - name: multi_hop_tracing_8k
      dataset: ruler
      data_dir: "8192"
      task_prefixes: ["vt"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - tova
        - streaming_llm
      budgets:
        default: [0.5]
"""
Path('configs/benchmark_sweeps.assignment_3.yaml').write_text(config_text, encoding='utf-8')
print(Path('configs/benchmark_sweeps.assignment_3.yaml').read_text())


In [ ]:
# Run Assignment 3.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.assignment_3.yaml


## Jamez: No Compression Baselines

Run the next two cells only. This assignment covers the `no_compression` baselines across all four categories and both context lengths.


In [ ]:
# Write the sweep config for Assignment 4.
from pathlib import Path

config_text = f"""
sweep:
  name: assignment_4
  model: {MODEL_NAME}
  device: auto
  torch_dtype: {TORCH_DTYPE}
  methods_config_path: configs/methods.yaml
  output_dir: {SHARED_RESULTS_DIR.as_posix()}/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: aggregation_4k_baseline
      dataset: ruler
      data_dir: "4096"
      task_prefixes: ["cwe", "fwe"]
      fraction: {FRACTION}
      methods: [no_compression]
      budgets:
        no_compression: [1.0]
"""
Path('configs/benchmark_sweeps.assignment_4.yaml').write_text(config_text, encoding='utf-8')
print(Path('configs/benchmark_sweeps.assignment_4.yaml').read_text())


In [ ]:
# Run Assignment 4.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.assignment_4.yaml


## Aarnav: Aggregation And Final Aggregation

Run the next two cells for the benchmark-backed `cwe, fwe` aggregation slice for the compressed methods at `4096` and `8192`. Then use the final aggregation cells after everyone has shared their outputs.


In [ ]:
# Write the sweep config for Assignment 5.
from pathlib import Path

config_text = f"""
sweep:
  name: assignment_5
  model: {MODEL_NAME}
  device: auto
  torch_dtype: {TORCH_DTYPE}
  methods_config_path: configs/methods.yaml
  output_dir: {SHARED_RESULTS_DIR.as_posix()}/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: aggregation_8k
      dataset: ruler
      data_dir: "8192"
      task_prefixes: ["cwe", "fwe"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - tova
        - streaming_llm
      budgets:
        default: [0.5]
"""
Path('configs/benchmark_sweeps.assignment_5.yaml').write_text(config_text, encoding='utf-8')
print(Path('configs/benchmark_sweeps.assignment_5.yaml').read_text())


In [ ]:
# Run Assignment 5.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.assignment_5.yaml


## Optional Environment Smoke Test

Use this only if you want to verify that a minimal benchmark-backed run works before launching the larger assignment sweeps.


In [ ]:
# Optional smoke test: one tiny benchmark-backed slice to verify the environment.
!python scripts/run_kvpress_benchmark_eval.py   --dataset ruler   --data-dir 4096   --model Qwen/Qwen2.5-1.5B-Instruct   --method no_compression   --budget 1.0   --fraction 0.002   --torch-dtype bfloat16   --output-dir "$SHARED_RESULTS_DIR/benchmark_eval"
